# baseline v4 (Qwen3.5-27B / non-thinking / A100 80GB)

이 베이스라인 코드는 `Qwen/Qwen3.5-27B`를 기준으로 `비추론(non-thinking) 모드`, `bf16 LoRA`, `배치 학습`, `파인튜닝`, `PEFT`를 적용한 버전입니다.

Colab의 **A100 80GB GPU** 환경을 권장합니다.
- 런타임 - 런타임 유형 변경 - GPU
- 가능하면 A100 80GB 사용


# 환경 준비

Qwen3.5-27B는 최신 `transformers`가 필요하므로 관련 라이브러리를 먼저 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작


In [ ]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" datasets pillow pandas --upgrade


In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [ ]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 압축 해제
!unzip "/content/drive/My Drive/2026-ssafy-15-2-ai.zip" -d "/content/"

# 라이브러리, 데이터, 설정

Qwen3.5-27B + bf16 LoRA + non-thinking 설정을 준비합니다.


In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Any
from transformers import (
    Qwen3_5ForConditionalGeneration,
    AutoProcessor,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen3.5-27B"
NON_THINKING = True
CHAT_TEMPLATE_KWARGS = {"enable_thinking": False} if NON_THINKING else {}

# A100 80GB 기준 권장 시작점 (full bf16 LoRA는 VRAM 사용량이 커서 보수적으로 시작)
IMAGE_SIZE = 384
MAX_LENGTH = 1536
MAX_NEW_TOKENS = 4

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM = 8
NUM_EPOCHS = 1
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2
IGNORE_INDEX = -100

# None이면 전체 학습, 숫자를 넣으면 스모크 테스트용 샘플링
DEBUG_TRAIN_SAMPLES = None

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

if DEBUG_TRAIN_SAMPLES is not None:
    sample_n = min(DEBUG_TRAIN_SAMPLES, len(train_df))
    train_df = train_df.sample(n=sample_n, random_state=SEED).reset_index(drop=True)

print("train size:", len(train_df))
print("test size :", len(test_df))


# 모델, Processor

Qwen3.5-27B는 대형 멀티모달 모델이므로 **bf16 LoRA**로 로드합니다.

- 기본값은 **비추론(non-thinking)** 모드
- A100 80GB 기준 보수적 시작 세팅 (`PER_DEVICE_BATCH_SIZE=1`, `GRAD_ACCUM=8`)
- OOM 발생 시 `MAX_LENGTH` 또는 `IMAGE_SIZE`를 먼저 낮추세요

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()


In [ ]:
# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"

# 사전학습 모델
base_model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# LoRA 학습을 위해 base model은 bf16으로 로드하고, gradient checkpointing으로 activation 메모리를 절약합니다.
if hasattr(base_model, "enable_input_require_grads"):
    base_model.enable_input_require_grads()

base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

# LoRA 세팅
# Qwen3.5 hybrid stack의 일반 attention / linear attention / MLP projection을 함께 학습
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "in_proj_qkv", "in_proj_z", "in_proj_a", "in_proj_b", "out_proj",
    ],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


# 프롬프트 템플릿

Qwen3.5-27B를 **비추론(non-thinking) 모드**로 사용하고, processor가 아니라 **tokenizer chat template**로 프롬프트를 구성해 thinking 스위치를 명시적으로 제어합니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()


In [ ]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one lowercase letter among a, b, c, or d. "
    "No reasoning. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )


# Custom Dataset, Collator

학습 시에는 **assistant 정답 토큰에만 loss**가 걸리도록 collator를 보강합니다.

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()


In [ ]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text}
            ]}
        ]

        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        full_texts, prefix_texts, images = [], [], []

        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            full_text = self.processor.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
                **CHAT_TEMPLATE_KWARGS
            )
            full_texts.append(full_text)
            images.append(img)

            if self.train:
                prefix_messages = messages[:-1]
                prefix_text = self.processor.tokenizer.apply_chat_template(
                    prefix_messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    **CHAT_TEMPLATE_KWARGS
                )
                prefix_texts.append(prefix_text)

        enc = self.processor(
            text=full_texts,
            images=images,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        if self.train:
            prefix_enc = self.processor(
                text=prefix_texts,
                images=images,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            )

            labels = enc["input_ids"].clone()
            labels[enc["attention_mask"] == 0] = IGNORE_INDEX

            for i in range(labels.size(0)):
                prefix_len = int(prefix_enc["attention_mask"][i].sum().item())
                labels[i, :prefix_len] = IGNORE_INDEX

            if "mm_token_type_ids" in enc:
                labels[enc["mm_token_type_ids"] != 0] = IGNORE_INDEX

            enc["labels"] = labels

        return enc


# DataLoader

A100 80GB 기준으로 `batch_size=2`, `grad_accum=4`부터 시작합니다.

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()


In [ ]:
# 검증용 데이터 분리
split = int(len(train_df) * 0.9)
train_subset = train_df.iloc[:split].reset_index(drop=True)
valid_subset = train_df.iloc[split:].reset_index(drop=True)

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 데이터로더
loader_kwargs = {
    "batch_size": PER_DEVICE_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": (device == "cuda"),
    "collate_fn": DataCollator(processor, True),
}

train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
valid_loader = DataLoader(valid_ds, shuffle=False, **loader_kwargs)

print("train batches:", len(train_loader))
print("valid batches:", len(valid_loader))


# fine-tuning

- Qwen3.5-27B + bf16 LoRA
- assistant 정답 토큰만 loss 계산
- A100 80GB 기준 시작 세팅

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()


In [ ]:
from contextlib import nullcontext
from tqdm.auto import tqdm

trainable_params = [p for p in model.parameters() if p.requires_grad]

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
total_update_steps = max(1, math.ceil(len(train_loader) * NUM_EPOCHS / GRAD_ACCUM))
warmup_steps = max(1, int(total_update_steps * WARMUP_RATIO))
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)

def autocast_context():
    if device == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()

global_step = 0
optimizer.zero_grad(set_to_none=True)

for epoch in range(NUM_EPOCHS):
    model.train()
    running = 0.0
    running_steps = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with autocast_context():
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        running += loss.item()
        running_steps += 1

        do_step = (step % GRAD_ACCUM == 0) or (step == len(train_loader))
        if do_step:
            torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            avg_loss = running / max(1, running_steps)
            progress_bar.set_postfix({
                "loss": f"{avg_loss:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}"
            })
            running = 0.0
            running_steps = 0

    model.eval()
    val_loss = 0.0
    val_steps = 0

    with torch.no_grad():
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k: v.to(device) for k, v in vb.items()}
            with autocast_context():
                val_loss += model(**vb).loss.item()
            val_steps += 1

    print(f"[Epoch {epoch+1}] valid loss {val_loss / max(1, val_steps):.4f}")

# 모델 저장
SAVE_DIR = "/content/qwen3_5_27b_lora_nonthinking_mcq"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


# inference

- 비추론(non-thinking) 모드 유지
- prompt를 제외한 **새로 생성된 토큰만** 디코딩
- 제출 파일 생성


In [ ]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"

    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = re.findall(r"[a-d]", last)
    if tokens:
        return tokens[-1]

    return "a"

# 추론을 위해 캐시 재활성화
model.config.use_cache = True
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": user_text}
        ]}
    ]

    text = processor.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_TEMPLATE_KWARGS
    )
    inputs = processor(
        text=[text],
        images=[img],
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            eos_token_id=processor.tokenizer.eos_token_id,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    generated_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    output_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]
    preds.append(extract_choice(output_text))

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")


In [ ]:
# 모델 응답 예시
print(output_text)
